# Example 01: Clear Preprocessing & Feature Selection Leakage (Type 1)

**Context**: Based on the classic Hastie & Tibshirani textbook scenario (ESL Ch 7.10.2: *'How Not to Do Cross-Validation'*).
High-dimensional tabular data with 200 samples and 1,000 features.

**Planted Flaw**: `StandardScaler` and `SelectKBest` are fitted on the entire dataset `X` before `train_test_split` is called.
**Ground Truth**: `CONFIRMED` Type 1 Leakage.
**Expected Outcome**: Leaked AUC ~0.76; Clean/Patched AUC ~0.50 (Delta ~ -0.25).

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
# Generate 200 samples with 1000 noisy/uninformative features
np.random.seed(42)
X = np.random.randn(200, 1000)
y = np.random.randint(0, 2, size=200)

In [ ]:
# FLAGGED LEAK: Preprocessing & feature selection fitted on entire dataset before split
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X_scaled, y)

In [ ]:
# Split happens AFTER preprocessing and feature selection
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.30, random_state=42
)

In [ ]:
clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_prob)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")